In [10]:
# 01 - Exploratory Data Analysis & Raw HRIS Data Audit

import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 20)

employees = pd.read_csv(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\raw\employees.csv")
compensation = pd.read_csv(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\raw\compensation.csv")
performance = pd.read_csv(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\raw\performance_reviews.csv")
exit_surveys = pd.read_csv(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\raw\exit_surveys.csv")
departments = pd.read_csv(r"C:\Users\ravi3\Desktop\repo_backup\Hr-people-analytics\data\raw\departments.csv")

print('employees:', employees.shape)
print('compensation:', compensation.shape)
print('performance_reviews:', performance.shape)
print('exit_surveys:', exit_surveys.shape)
print('departments:', departments.shape)

employees.head(5)


employees: (3347, 14)
compensation: (3347, 4)
performance_reviews: (8391, 5)
exit_surveys: (991, 7)
departments: (7, 3)


,employee_id,department,job_role,job_level,gender,age,education_level,marital_status,hire_date,termination_date,termination_reason,active_status,distance_from_home_miles,remote_flag
0,EMP-11829,Operations,Facilities Coordinator,1,Male,37,Bachelor's,Divorced,2022-08-25,NaN,NaN,1,1.5,False
1,EMP-12626,Finance,Financial Analyst,1,Female,28,Master's,Single,2022-11-20,NaN,NaN,1,42.5,False
2,EMP-11551,Customer Success,Customer Success Rep,1,Female,35,Bachelor's,Single,2022-06-10,NaN,NaN,1,7.6,False
3,EMP-11236,Customer Success,CS Team Lead,3,Female,43,Bachelor's,Single,2021-05-12,2024-03-14,Better Opportunity,0,0.5,False
4,EMP-12973,Engineering,Engineering Manager,3,Male,32,Master's,Married,2024-10-06,2025-05-18,Better Opportunity,0,1.0,True


In [11]:
## Cell 2: Issue 1 - Duplicate Employee Records (HRIS Migration Glitch)

dupe_count = employees['employee_id'].duplicated().sum()
print(f"{dupe_count} duplicate employee_id rows (re-ingested during an HRIS migration).")

dupe_mask = employees['employee_id'].duplicated(keep=False)
employees[dupe_mask].sort_values('employee_id').head(6)


47 duplicate employee_id rows (re-ingested during an HRIS migration).


,employee_id,department,job_role,job_level,gender,age,education_level,marital_status,hire_date,termination_date,termination_reason,active_status,distance_from_home_miles,remote_flag
22,EMP-10020,Engineering,Software Engineer,1,Non-binary,39,Master's,Married,2023-09-10,NaN,NaN,1,13.9,False
1673,EMP-10020,Engineering,Software Engineer,1,Non-binary,39,Master's,Married,2023-09-10,NaN,NaN,1,13.9,False
1315,EMP-10384,Engineering,Senior Software Engineer,2,Female,43,Bachelor's,Married,2022-03-09,NaN,NaN,1,2.5,True
2607,EMP-10384,Engineering,Senior Software Engineer,2,Female,43,Bachelor's,Married,2022-03-09,NaN,NaN,1,2.5,True
1177,EMP-10440,engineering,QA Engineer,1,Male,34,Bachelor's,Married,2024-03-10,NaN,NaN,0,0.6,True
874,EMP-10440,Engineering,QA Engineer,1,Male,34,Bachelor's,Married,2024-03-10,NaN,NaN,1,0.6,True


In [12]:
## Cell 3: Issue 2 - Inconsistent Department Labels (27 Raw Variants)

print("Unique department label variants in raw data:", employees['department'].nunique())
employees['department'].value_counts().head(12)


Unique department label variants in raw data: 27


department
Engineering         1083
Sales                618
Customer Success     390
Marketing            320
Finance              276
Operations           185
HR                   181
 Engineering          38
ENGINEERING           35
sales                 25
engineering           25
SALES                 21
Name: count, dtype: int64

In [13]:
## Cell 4: Issue 3 - Status-Sync Gap (active_status = 0 but No termination_date)

gap = (employees['active_status'] == 0) & (employees['termination_date'].isna())
print(f"{gap.sum()} employees marked departed with no termination_date recorded.")
employees[gap][['employee_id', 'department', 'hire_date', 'termination_date', 'active_status']].head(5)


125 employees marked departed with no termination_date recorded.


,employee_id,department,hire_date,termination_date,active_status
14,EMP-12878,Sales,2022-10-11,NaN,0
21,EMP-10511,ENGINEERING,2024-02-19,NaN,0
39,EMP-11738,HR,2022-10-14,NaN,0
70,EMP-12314,Marketing,2022-01-30,NaN,0
74,EMP-10183,Marketing,2021-12-01,NaN,0


In [14]:
## Cell 5: Issue 4 - Chronological Inversion (hire_date AFTER termination_date)

emp_dates = employees.copy()
emp_dates['hire_date'] = pd.to_datetime(emp_dates['hire_date'], errors='coerce')
emp_dates['termination_date'] = pd.to_datetime(emp_dates['termination_date'], errors='coerce')

swapped = emp_dates['termination_date'].notna() & (emp_dates['hire_date'] > emp_dates['termination_date'])
print(f"{swapped.sum()} rows have hire_date AFTER termination_date.")
emp_dates[swapped][['employee_id', 'hire_date', 'termination_date']].head(5)


33 rows have hire_date AFTER termination_date.


,employee_id,hire_date,termination_date
65,EMP-11242,2025-12-23,2022-02-12
159,EMP-12357,2023-07-31,2022-09-03
221,EMP-10776,2023-03-20,2023-02-07
254,EMP-10144,2025-04-20,2022-07-28
419,EMP-10443,2024-12-26,2021-04-13


In [15]:
## Cell 6: Issues 5 & 6 - Invalid Commute Distance (<= 0) & Invalid Base Salary (<= 0)

bad_dist = (employees['distance_from_home_miles'] <= 0).sum()
print(f"Issue 5: {bad_dist} rows have distance_from_home_miles <= 0 (GPS/geocoding glitch).")

bad_sal = (compensation['base_salary'] <= 0).sum()
print(f"Issue 6: {bad_sal} compensation rows have base_salary <= 0 (payroll sync glitch).")


Issue 5: 58 rows have distance_from_home_miles <= 0 (GPS/geocoding glitch).
Issue 6: 61 compensation rows have base_salary <= 0 (payroll sync glitch).


In [16]:
## Cell 7: Issue 7 - Performance Scores Outside the Valid 1-5 Scale

bad_perf = (~performance['performance_score'].between(1, 5)).sum()
print(f"{bad_perf} performance review rows have a score outside 1-5.")
performance[~performance['performance_score'].between(1, 5)].head(5)


41 performance review rows have a score outside 1-5.


,review_id,employee_id,review_date,performance_score,promotion_flag
165,166,EMP-10051,2023-12-28,6,False
406,407,EMP-10805,2025-07-06,7,False
444,445,EMP-12303,2023-03-03,7,False
567,568,EMP-11697,2024-06-03,0,False
609,610,EMP-10670,2021-06-20,0,False


In [17]:
## Cell 8: Issue 8 - Orphaned Exit-Survey Rows (No Matching Employee)

orphans = ~exit_surveys['employee_id'].isin(employees['employee_id'])
print(f"{orphans.sum()} exit survey rows reference an employee_id that does not exist in the employees table.")
exit_surveys[orphans].head(5)


24 exit survey rows reference an employee_id that does not exist in the employees table.


,exit_survey_id,employee_id,exit_date,reason,satisfaction_score,work_life_balance_score,would_recommend
967,968,EMP-99000,2021-06-15,Other,1,3,False
968,969,EMP-99001,2021-06-15,Other,1,3,False
969,970,EMP-99002,2021-06-15,Other,4,5,False
970,971,EMP-99003,2021-06-15,Other,2,2,False
971,972,EMP-99004,2021-06-15,Other,1,1,True
